# Syed Huzaifa Rahman 24I-2500 DS-A

## Assignment 2 Game UNO

### PLAYERS:
### Player 1 (AI Minimax Defensive)
### Player 2 (AI / Expectimax Offensive)
### Player 3 (User /AI Manual or Simulation)

In [10]:
# 1 push 
import numpy as np
import random

colors = ["Red", "Blue", "Green", "Yellow"]
skip = "skip"

class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value
        self._validate()

    def _validate(self):
        if type(self.color) is not str:
            raise TypeError("Card color must be a string")
        if self.color not in colors:
            raise ValueError("Invalid color: " + str(self.color))

        if type(self.value) is int:
            if self.value < 0 or self.value > 9:
                raise ValueError("Card number must be 0 to 9")
        elif type(self.value) is str:
            if self.value != skip:
                raise ValueError("Invalid special value: " + str(self.value))
        else:
            raise TypeError("Card value must be int or 'skip'")

    def is_skip(self):
        return type(self.value) is str and self.value == skip

    def matches(self, top_card):
        # same color or number
        if self.color == top_card.color:
            return True
        if self.is_skip() or top_card.is_skip():
            return False
        return self.value == top_card.value

    def __str__(self):
        return self.color + " " + str(self.value)


class GameState:
    def __init__(self, hands, top_card, deck, current_player):
        self.hands = hands
        self.top_card = top_card
        self.deck = deck
        # 0,1,2
        self.current_player = current_player

    def clone(self):
        new_hands = []
        i = 0
        while i < 3:
            new_hands.append(list(self.hands[i]))
            i += 1
        return GameState(new_hands, self.top_card, list(self.deck), self.current_player)

    def is_terminal(self):
        i = 0
        while i < 3:
            if len(self.hands[i]) == 0:
                return True
            i += 1
        return False


def generate_deck():
    # deck has 0 to 9 of every color and a skip then shiffled
    deck = []
    i = 0
    while i < len(colors):
        c = colors[i]
        v = 0
        while v <= 9:
            deck.append(Card(c, v))
            v += 1
        deck.append(Card(c, skip))
        i += 1

    # shuffle using constraint
    perm = np.random.permutation(len(deck))
    shuffled = []
    k = 0
    while k < len(perm):
        shuffled.append(deck[int(perm[k])])
        k += 1
    return shuffled


def _take_last(deck):
    # pick/draw card
    card = deck[len(deck) - 1]
    del deck[len(deck) - 1]
    return card


def deal_initial_state():
    # every player gets 5 non skip cards
    deck = generate_deck()
    hands = [[], [], []]

    r = 0
    while r < 5:
        p = 0
        while p < 3:
            hands[p].append(_take_last(deck))
            p += 1
        r += 1

    while True:
        top = _take_last(deck)
        if not top.is_skip():
            top_card = top
            break
        deck.insert(0, top)

    return GameState(hands, top_card, deck, 0)


# checking random cards and shuffle here (state change every time cell runs)
_state0 = deal_initial_state()
print("Top card:", str(_state0.top_card))
print("Hand sizes:", [len(h) for h in _state0.hands], "Deck:", len(_state0.deck))

Top card: Green 5
Hand sizes: [5, 5, 5] Deck: 28


In [11]:
# Legal moves and State transition
# push 1
def get_valid_moves(hand, top_card):
    plays = []
    i = 0
    while i < len(hand):
        if hand[i].matches(top_card):
            plays.append(("play", hand[i]))
        i += 1
    if len(plays) > 0:
        return plays
        
    return [("draw", None)]

def _remove_first_occurrence(hand, card):
    i = 0
    while i < len(hand):
        if hand[i].color == card.color and hand[i].value == card.value:
            del hand[i]
            return True
        i += 1
    return False

def apply_move(state, move):
    new_state = state.clone()
    player = new_state.current_player
    hand = new_state.hands[player]
    action = move[0]
    card = move[1]
    if action == "play":
        if card is None:
            raise ValueError("Play move requires a card")
            
        if not card.matches(new_state.top_card):
            raise ValueError("Illegal move")

        ok = _remove_first_occurrence(hand, card)
        if not ok:
            raise ValueError("Card not found in hand")

        new_state.top_card = card
        # skip skips next player's turn
        if card.is_skip():
            new_state.current_player = (player + 2) % 3
        else:
            new_state.current_player = (player + 1) % 3
        return new_state

    if action == "draw":
        # Rule 2 draw exactly 1 if no valid move
        if len(new_state.deck) > 0:
            hand.append(_take_last(new_state.deck))
            
        new_state.current_player = (player + 1) % 3
        return new_state
    raise ValueError("Invalid move action")

# quick check to see for every time program runs 
hand = _state0.hands[_state0.current_player]
moves = get_valid_moves(hand, _state0.top_card)
print("Current player:", _state0.current_player)
print("Valid moves:", [m[0] if m[0] == "draw" else str(m[1]) for m in moves])

Current player: 0
Valid moves: ['Green 8', 'Yellow 5']


In [12]:
# Eval funcs defensive/offensive
# push 2
def eval_base(state: GameState, player_index: int) -> float:
    c_ai = float(len(state.hands[player_index]))
    opp_cards = []
    for i in range(3):
        if i != player_index:
            opp_cards.append(len(state.hands[i]))
    c_opp = float(sum(opp_cards)) / 2.0
    s = 0.0
    for card in state.hands[player_index]:
        if card.is_skip():
            s += 1.0
    return 50.0 - 5.0 * c_ai + 2.0 * c_opp + 3.0 * s

def eval_defensive(state: GameState, player_index: int) -> float:
    # we prioritize blocking opps that are close to winning
    score = eval_base(state, player_index)
    for i in range(3):
        if i == player_index:
            continue
        c = len(state.hands[i])
        if c <= 2:
            # extra penalty if opps are near 0
            score -= 8.0 * float(3 - c)
    return score

def eval_offensive(state: GameState, player_index: int) -> float:
    # prioritize remove cards more
    c_ai = float(len(state.hands[player_index]))
    opp_cards = []
    for i in range(3):
        if i != player_index:
            opp_cards.append(len(state.hands[i]))
    c_opp = float(sum(opp_cards)) / 2.0
    s = 0.0
    for card in state.hands[player_index]:
        if card.is_skip():
            s += 1.0
    return 50.0 - 7.0 * c_ai + 1.0 * c_opp + 4.0 * s

print("Baseline score for P1:", eval_base(_state0, 0))
print("Defensive score for P1:", eval_defensive(_state0, 0))
print("Offensive score for P2:", eval_offensive(_state0, 1))

Baseline score for P1: 35.0
Defensive score for P1: 35.0
Offensive score for P2: 24.0


In [13]:
# minimax depth =3 and expectimax depth=3

# push 3

def minimax(state, depth, max_player, eval_fn):
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player)), None
    current = state.current_player
    moves = get_valid_moves(state.hands[current], state.top_card)

    if current == max_player:
        best_val = -10000
        best_move = None
        i = 0
        while i < len(moves):
            child = apply_move(state, moves[i])
            val, _ = minimax(child, depth - 1, max_player, eval_fn)
            if val > best_val:
                best_val = val
                best_move = moves[i]
            i += 1
        return best_val, best_move

    best_val = 10000
    best_move = None
    i = 0
    while i < len(moves):
        child = apply_move(state, moves[i])
        val, _ = minimax(child, depth - 1, max_player, eval_fn)
        if val < best_val:
            best_val = val
            best_move = moves[i]
        i += 1
    return best_val, best_move


def _remove_deck_index(deck, idx):
    # remove value at deck[idx]
    card = deck[idx]
    del deck[idx]
    return card


def expectimax(state, depth, max_player, eval_fn):
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player))
    current = state.current_player
    moves = get_valid_moves(state.hands[current], state.top_card)
    # max node
    if current == max_player:
        best_val = -10000
        i = 0
        while i < len(moves):
            mv = moves[i]
            if mv[0] == "draw":
                val = expectimax_chance_node(state, depth, max_player, eval_fn)
            else:
                child = apply_move(state, mv)
                val = expectimax(child, depth - 1, max_player, eval_fn)

            if val > best_val:
                best_val = val
            i += 1
        return best_val

    # opp node which is a random legal move
    mv = random.choice(moves)
    if mv[0] == "draw":
        return expectimax_chance_node(state, depth, max_player, eval_fn)
    child = apply_move(state, mv)
    return expectimax(child, depth - 1, max_player, eval_fn)


def expectimax_chance_node(state, depth, max_player, eval_fn):
    # chance node where you draw one card randomly from deck
    if depth == 0 or state.is_terminal():
        return float(eval_fn(state, max_player))
    if len(state.deck) == 0:
        return float(eval_fn(state, max_player))

    total = float(len(state.deck))
    expected = 0.0
    i = 0
    while i < len(state.deck):
        child = state.clone()
        player = child.current_player
        drawn = _remove_deck_index(child.deck, i)
        child.hands[player].append(drawn)
        child.current_player = (player + 1) % 3
        val = expectimax(child, depth - 1, max_player, eval_fn)
        expected += (1.0 / total) * val
        i += 1
    return expected


def choose_move_minimax(state, player_index):
    _, mv = minimax(state, 3, player_index, eval_defensive)
    if mv is None:
        return get_valid_moves(state.hands[player_index], state.top_card)[0]
    return mv


def choose_move_expectimax(state, player_index):
    moves = get_valid_moves(state.hands[player_index], state.top_card)
    best_mv = moves[0]
    best_val = -10000
    i = 0
    while i < len(moves):
        mv = moves[i]
        if mv[0] == "draw":
            val = expectimax_chance_node(state, 3, player_index, eval_offensive)
        else:
            child = apply_move(state, mv)
            val = expectimax(child, 2, player_index, eval_offensive)
        if val > best_val:
            best_val = val
            best_mv = mv
        i += 1
    return best_mv


print("Minimax suggested move for P1:", choose_move_minimax(_state0, 0))

# quick check to see expectimax for P2's turn
_tmp = _state0.clone()
_tmp.current_player = 1
print("Expectimax suggested move for P2:", choose_move_expectimax(_tmp, 1))

Minimax suggested move for P1: ('play', <__main__.Card object at 0x13ad0c4d0>)
Expectimax suggested move for P2: ('play', <__main__.Card object at 0x13acb9d90>)


In [14]:
# simulation showing proper running and printing like maam did in pdf
# last PUSH 4
def _print_move(player, move):
    if move[0] == "draw":
        print(f"Player {player + 1} -> Draw")
    else:
        print(f"Player {player + 1} -> Play: {move[1]}")


def choose_move_expectimax_with_print(state, player_index, depth=3):
    print("Top card:", str(state.top_card))
    print("AI hand:")
    for c in state.hands[player_index]:
        print(str(c))

    print("AI decision(All possible decisions considered at depth 1):")
    print("")
    moves = get_valid_moves(state.hands[player_index], state.top_card)
    # game tree example
    print("\nGENERATED SEARCH TREE (EXAMPLE RUN) ---")
    print("\n          AI (Root)")
    print("         /    |    \\")
    move_names = []
    i = 0
    while i < len(moves):
        if moves[i][0] == "play":
            move_names.append(str(moves[i][1]))
        else:
            move_names.append("Draw")
        i += 1
    print(f"  {'   '.join(move_names)}")
    print("       /      |      \\")
    print("  Opponent  Opponent  Chance\n")
    best_mv = moves[0]
    best_val = -10000
    for mv in moves:
        if mv[0] == "draw":
            val = expectimax_chance_node(state, depth, player_index, eval_offensive)
            print("Draw a card")
            print("Expected score:", round(val, 2))
        else:
            child = apply_move(state, mv)
            val = expectimax(child, depth - 1, player_index, eval_offensive)
            print("Play:", str(mv[1]))
            print("Expected score:", round(val, 2))
        if val > best_val:
            best_val = val
            best_mv = mv
            
    return best_mv
    
def user_choose_move(state, player_index):
    hand = state.hands[player_index]
    moves = get_valid_moves(hand, state.top_card)
    if len(moves) == 1 and moves[0][0] == "draw":
        print("No valid moves. You must draw 1 card.")
        input("Press Enter to draw...")
        return ("draw", None)

    while True:
        print("Top card:", str(state.top_card))
        print("Choose a card index to play (0-based):")
        for i, card in enumerate(hand):
            valid = any(mv[0] == "play" and mv[1] == card for mv in moves)
            tag = "(valid)" if valid else "(invalid)"
            print(i, ":", str(card), tag)
        choice = input("Your choice: ").strip()
        try:
            idx = int(choice)
        except ValueError:
            print("Invalid input.")
            continue
        if idx < 0 or idx >= len(hand):
            print("Index out of range.")
            continue
        selected = hand[idx]
        if not any(mv[0] == "play" and mv[1] == selected for mv in moves):
            print("Illegal move.")
            continue
        return ("play", selected)

def simulate_game(player3_mode):
    # simulate all moves to end
    state = deal_initial_state()
    turn = 0

    while not state.is_terminal():
        turn += 1
        p = state.current_player

        print("\n" + "=" * 50)
        print(f"Turn {turn} | Current Player: {p + 1}")
        print("Top card:", state.top_card)
        print("Hand sizes:", [len(h) for h in state.hands], "Deck:", len(state.deck))
        if p == 0:
            mv = choose_move_minimax(state, 0)
            _print_move(0, mv)
            state = apply_move(state, mv)
            continue
        if p == 1:
            mv = choose_move_expectimax_with_print(state, 1, depth=3)
            print("Chosen move:", "Draw" if mv[0] == "draw" else str(mv[1]))
            state = apply_move(state, mv)
            continue
        # Player 3
        if player3_mode == "manual":
            mv = user_choose_move(state, 2)
        else:
            mv = choose_move_minimax(state, 2)
        _print_move(2, mv)
        state = apply_move(state, mv)

    for i in range(3):
        if len(state.hands[i]) == 0:
            print("\nGame over! Player", i + 1, "wins!")
            return i

    return -1


In [15]:
# run the game
# last PUSH 4
mode = input("Choose mode for Player 3 (manual/sim): ").strip().lower()
player3_mode = "ai" if mode == "sim" else "manual"
simulate_game(player3_mode)

Choose mode for Player 3 (manual/sim):  sim



Turn 1 | Current Player: 1
Top card: Yellow 6
Hand sizes: [5, 5, 5] Deck: 28
Player 1 -> Play: Yellow 4

Turn 2 | Current Player: 2
Top card: Yellow 4
Hand sizes: [4, 5, 5] Deck: 28
Top card: Yellow 4
AI hand:
Green 8
Red 5
Blue 1
Blue 3
Red 3
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Draw
       /      |      \
  Opponent  Opponent  Chance

Draw a card
Expected score: 12.39
Chosen move: Draw

Turn 3 | Current Player: 3
Top card: Yellow 4
Hand sizes: [4, 6, 5] Deck: 27
Player 3 -> Play: Yellow 5

Turn 4 | Current Player: 1
Top card: Yellow 5
Hand sizes: [4, 6, 4] Deck: 27
Player 1 -> Draw

Turn 5 | Current Player: 2
Top card: Yellow 5
Hand sizes: [5, 6, 4] Deck: 26
Top card: Yellow 5
AI hand:
Green 8
Red 5
Blue 1
Blue 3
Red 3
Red skip
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Red 5


1

Player 3 -> Play: Green 6

Turn 4 | Current Player: 1
Top card: Green 6
Hand sizes: [4, 6, 4] Deck: 27
Player 1 -> Play: Green 1

Turn 5 | Current Player: 2
Top card: Green 1
Hand sizes: [3, 6, 4] Deck: 27
Top card: Green 1
AI hand:
Yellow 5
Yellow 7
Red 0
Blue 0
Red 3
Red 7
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Draw
       /      |      \
  Opponent  Opponent  Chance

Draw a card
Expected score: 6.09
Chosen move: Draw

Turn 6 | Current Player: 3
Top card: Green 1
Hand sizes: [3, 7, 4] Deck: 26
No valid moves. You must draw 1 card.


Press Enter to draw... 


...

Player 3 -> Draw

Turn 46 | Current Player: 1
Top card: Red 0
Hand sizes: [3, 3, 2] Deck: 9
Player 1 -> Draw

Turn 47 | Current Player: 2
Top card: Red 0
Hand sizes: [4, 3, 2] Deck: 8
Top card: Red 0
AI hand:
Yellow 5
Yellow skip
Red 6
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Red 6
       /      |      \
  Opponent  Opponent  Chance

Play: Red 6
Expected score: 42.0
Chosen move: Red 6

Turn 48 | Current Player: 3
Top card: Red 6
Hand sizes: [4, 2, 2] Deck: 8
Top card: Red 6
Choose a card index to play (0-based):
0 : Blue 6 (valid)
1 : Yellow 9 (invalid)


Your choice:  0


Player 3 -> Play: Blue 6

Turn 49 | Current Player: 1
Top card: Blue 6
Hand sizes: [4, 2, 1] Deck: 8
Player 1 -> Play: Yellow 6

Turn 50 | Current Player: 2
Top card: Yellow 6
Hand sizes: [3, 2, 1] Deck: 8
Top card: Yellow 6
AI hand:
Yellow 5
Yellow skip
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Yellow 5   Yellow skip
       /      |      \
  Opponent  Opponent  Chance

Play: Yellow 5
Expected score: 48.5
Play: Yellow skip
Expected score: 51.5
Chosen move: Yellow skip

Turn 51 | Current Player: 1
Top card: Yellow skip
Hand sizes: [3, 1, 1] Deck: 8
Player 1 -> Play: Yellow 3

Turn 52 | Current Player: 2
Top card: Yellow 3
Hand sizes: [2, 1, 1] Deck: 8
Top card: Yellow 3
AI hand:
Yellow 5
AI decision(All possible decisions considered at depth 1):


GENERATED SEARCH TREE (EXAMPLE RUN) ---

          AI (Root)
         /    |    \
  Yellow 5
       /      |      \
  Opponent  Opponent  Ch

1

0

In [16]:
# BONUS: Visual simulation (Tkinter)

import tkinter as tk
from tkinter import scrolledtext


CARD_BG = {
    "Red": "#d94848",
    "Blue": "#3b82f6",
    "Green": "#22c55e",
    "Yellow": "#eab308"
}


class UNOVisualSimulator:
    def __init__(self, root):
        self.root = root
        self.root.title("UNO AI Visual Simulation")
        self.root.configure(bg="#25364f")

        self.state = deal_initial_state()
        self.game_over = False
        self.turn_count = 0

        # Title
        self.title_lbl = tk.Label(
            root,
            text="UNO AI Visual Simulation",
            font=("Arial", 16, "bold"),
            fg="#facc15",
            bg="#25364f"
        )
        self.title_lbl.pack(pady=8)

        # Turn info
        self.turn_lbl = tk.Label(
            root,
            text="Current Turn: Player 1",
            font=("Arial", 12, "bold"),
            fg="white",
            bg="#25364f"
        )
        self.turn_lbl.pack(pady=4)

        # Winner banner (shown only when game ends)
        self.winner_lbl = tk.Label(
            root,
            text="",
            font=("Arial", 20, "bold"),
            fg="#facc15",
            bg="#25364f"
        )
        self.winner_lbl.pack(pady=6)

        self.game_over_btn = tk.Button(
            root,
            text="Game Over",
            command=self.reset_game,
            bg="#64748b",
            fg="white",
            font=("Arial", 12, "bold"),
            padx=20,
            pady=8
        )
        # hidden initially; shown when terminal state reached
        self.game_over_btn.pack(pady=4)
        self.game_over_btn.pack_forget()

        # Board area
        self.board = tk.Frame(root, bg="#25364f")
        self.board.pack(fill="both", expand=True, padx=10, pady=10)

        self.p1_frame = tk.LabelFrame(self.board, text="Player 1 (Defensive)", fg="white", bg="#25364f")
        self.p1_frame.grid(row=0, column=0, columnspan=3, pady=10)

        self.p2_frame = tk.LabelFrame(self.board, text="Player 2 (Offensive)", fg="white", bg="#25364f")
        self.p2_frame.grid(row=1, column=0, padx=12, pady=10, sticky="w")

        self.center_frame = tk.LabelFrame(self.board, text="Top Card", fg="white", bg="#25364f")
        self.center_frame.grid(row=1, column=1, padx=12, pady=10)

        self.p3_frame = tk.LabelFrame(self.board, text="Player 3 (Defensive)", fg="white", bg="#25364f")
        self.p3_frame.grid(row=1, column=2, padx=12, pady=10, sticky="e")

        # Controls
        self.controls = tk.Frame(root, bg="#1e293b")
        self.controls.pack(fill="x", pady=6)

        self.next_btn = tk.Button(
            self.controls,
            text="Play Next Turn",
            command=self.play_next_turn,
            bg="#f59e0b",
            fg="black",
            font=("Arial", 11, "bold"),
            padx=10,
            pady=6
        )
        self.next_btn.pack(side="left", padx=10, pady=8)

        self.reset_btn = tk.Button(
            self.controls,
            text="Reset Game",
            command=self.reset_game,
            bg="#94a3b8",
            fg="black",
            font=("Arial", 11, "bold"),
            padx=10,
            pady=6
        )
        self.reset_btn.pack(side="left", padx=8, pady=8)

        # Log area
        self.log = scrolledtext.ScrolledText(root, height=10, width=110, bg="#0f172a", fg="#e2e8f0")
        self.log.pack(fill="both", padx=10, pady=8)

        self.log_message("Game initialized. Click 'Play Next Turn' to start.")
        self.render()

    def log_message(self, msg):
        self.log.insert(tk.END, msg + "\n")
        self.log.see(tk.END)

    def clear_frame(self, frame):
        for widget in frame.winfo_children():
            widget.destroy()

    def create_card_widget(self, parent, card):
        color = CARD_BG.get(card.color, "#64748b")
        text = str(card.value)
        if card.is_skip():
            text = "SKIP"

        # A simple UNO-like tile
        box = tk.Frame(parent, bg="white", bd=2, relief="solid")
        box.pack(side="left", padx=5, pady=5)

        lbl = tk.Label(
            box,
            text=text,
            width=6,
            height=3,
            bg=color,
            fg="black",
            font=("Arial", 11, "bold"),
        )
        lbl.pack(padx=3, pady=3)

    def create_top_card_widget(self, parent, card):
        color = CARD_BG.get(card.color, "#64748b")
        text = str(card.value)
        if card.is_skip():
            text = "SKIP"

        # Larger card for center display
        box = tk.Frame(parent, bg="white", bd=3, relief="solid")
        box.pack(padx=10, pady=10)

        lbl = tk.Label(
            box,
            text=text,
            width=7,
            height=4,
            bg=color,
            fg="black",
            font=("Arial", 18, "bold"),
        )
        lbl.pack(padx=8, pady=8)

    def render_player_hand(self, frame, hand):
        self.clear_frame(frame)
        i = 0
        while i < len(hand):
            self.create_card_widget(frame, hand[i])
            i += 1

    def render_top_card(self):
        self.clear_frame(self.center_frame)
        self.create_top_card_widget(self.center_frame, self.state.top_card)

    def render(self):
        self.turn_lbl.config(text="Current Turn: Player " + str(self.state.current_player + 1))
        self.render_player_hand(self.p1_frame, self.state.hands[0])
        self.render_player_hand(self.p2_frame, self.state.hands[1])
        self.render_player_hand(self.p3_frame, self.state.hands[2])
        self.render_top_card()

    def get_ai_move(self, player):
        if player == 0:
            return choose_move_minimax(self.state, 0)
        if player == 1:
            return choose_move_expectimax(self.state, 1)
        return choose_move_minimax(self.state, 2)

    def play_next_turn(self):
        if self.game_over:
            self.log_message("Game already finished. Press Reset Game to play again.")
            return

        self.turn_count += 1
        p = self.state.current_player
        move = self.get_ai_move(p)

        move_text = "Draw"
        if move[0] == "play":
            move_text = "Play " + str(move[1])

        self.log_message(
            "Turn " + str(self.turn_count) +
            " | Player " + str(p + 1) +
            " | Top " + str(self.state.top_card) +
            " | Action: " + move_text
        )

        self.state = apply_move(self.state, move)

        # terminal check
        i = 0
        while i < 3:
            if len(self.state.hands[i]) == 0:
                self.game_over = True
                # winner banner
                roles = {0: "Defensive", 1: "Offensive", 2: "Defensive"}
                self.winner_lbl.config(text="WINNER: Player " + str(i + 1) + " (" + roles[i] + ")")
                self.game_over_btn.pack(pady=10)
                self.render()
                self.log_message("GAME OVER -> Player " + str(i + 1) + " wins!")
                self.next_btn.config(state="disabled")
                return
            i += 1

        self.render()

    def reset_game(self):
        self.state = deal_initial_state()
        self.game_over = False
        self.turn_count = 0
        self.next_btn.config(state="normal")
        self.winner_lbl.config(text="")
        self.game_over_btn.pack_forget()
        self.log.delete("1.0", tk.END)
        self.log_message("Game reset. Click 'Play Next Turn' to start.")
        self.render()


def launch_uno_visual_simulation():
    root = tk.Tk()
    root.geometry("1200x760")
    app = UNOVisualSimulator(root)
    root.mainloop()

launch_uno_visual_simulation()